# Pipeline Analisis Sentimen: Review Aplikasi Gojek

Notebook ini mendemonstrasikan alur kerja lengkap untuk analisis sentimen pada dataset review aplikasi Gojek. Prosesnya mencakup:

1.  **Setup & Instalasi**: Mempersiapkan environment dan menginstall library yang dibutuhkan.
2.  **Memuat Data**: Mengimpor dataset review.
3.  **Preprocessing Teks**: Membersihkan dan mempersiapkan data teks menggunakan `NLPProcessor`.
4.  **Ekstraksi Fitur**: Mengubah teks menjadi fitur numerik menggunakan Bag-of-Words (BoW) dan TF-IDF.
5.  **Training Model**: Melatih tiga model klasifikasi: Naive Bayes, Decision Tree, dan SVM.
6.  **Evaluasi Model**: Mengevaluasi performa model menggunakan metrik standar (Akurasi, Presisi, Recall, F1) dan Confusion Matrix.
7.  **Visualisasi**: Menampilkan hasil evaluasi secara visual.
8.  **Prediksi**: Menggunakan model terbaik untuk memprediksi sentimen pada teks baru.

---

### Catatan Penting Mengenai Peran Notebook Ini

Notebook ini berfungsi sebagai **lingkungan pengembangan, prototipe, dan dokumentasi teknis** dari seluruh alur kerja analisis sentimen. Semua logika yang Anda lihat di sini (mulai dari preprocessing hingga training model) telah diimplementasikan ke dalam aplikasi web interaktif.

**Perbedaan Utama dengan Aplikasi Web:**

*   **Aplikasi Web (direktori `frontend` dan `backend`):**
    *   Melakukan training & evaluasi secara *real-time* melalui antarmuka.
    *   **Secara otomatis menyimpan model terbaik** ke disk (`backend/models/best_model.joblib`).
    *   **Memuat ulang model secara otomatis** saat server dimulai, sehingga siap digunakan untuk prediksi tanpa training ulang.

*   **Notebook ini:**
    *   Digunakan untuk eksperimen awal dan validasi metode.
    *   Hasilnya (model yang dilatih) **tidak disimpan secara otomatis**.

**Rekomendasi:** Untuk penggunaan dan demonstrasi, jalankan aplikasi web menggunakan skrip `run.sh` atau `run.bat`. Gunakan notebook ini untuk memahami detail setiap langkah proses secara mendalam.

---

## 1. Setup & Instalasi

Sel pertama ini akan menginstall semua dependensi yang diperlukan yang tercantum dalam `backend/requirements.txt`.

In [ ]:
%pip install -r backend/requirements.txt

## 2. Impor Library & Konfigurasi Awal

Sekarang, kita mengimpor semua library yang akan digunakan dan menambahkan direktori `backend` ke path sistem agar kita bisa mengimpor `NLPProcessor`.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Tambahkan direktori backend ke path
sys.path.append(os.path.abspath(os.path.join('.', 'backend')))

# Impor kelas dari backend
from nlp_processor import NLPProcessor

# Impor komponen dari Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

print("Setup selesai. Library berhasil diimpor.")

## 3. Memuat Data

Kita akan memuat dataset `GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv` dan melihat beberapa baris pertama untuk memahami strukturnya.

In [ ]:
DATASET_PATH = 'datasets/GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv'

try:
    df = pd.read_csv(DATASET_PATH)
    print("Dataset berhasil dimuat.")
    print(f"Jumlah baris: {len(df)}")
    print("Kolom yang tersedia:", df.columns.tolist())
    df.head()

Kita perlu memastikan kolom `sentiment` dan `review` ada di dalam dataset.

In [ ]:
TEXT_COLUMN = 'review'
LABEL_COLUMN = 'sentiment'

if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
    raise ValueError(f"Kolom '{TEXT_COLUMN}' atau '{LABEL_COLUMN}' tidak ditemukan di dataset.")

# Hapus baris dengan nilai null di kolom teks atau label
df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)

print("Distribusi Sentimen:")
print(df[LABEL_COLUMN].value_counts())

## 4. Preprocessing Teks

Di tahap ini, kita akan menggunakan `NLPProcessor` untuk membersihkan data teks. Ini termasuk case folding, menghapus URL, mention, emoji, tokenisasi, menghapus stopword, dan stemming.

In [ ]:
# Inisialisasi NLP Processor
nlp = NLPProcessor()

# Contoh preprocessing pada satu teks
sample_text = df[TEXT_COLUMN].iloc[0]
print("Teks Asli:", sample_text)

# Tanpa stemming
processed_sample = nlp.preprocess(sample_text, apply_stemming=False)
print("
--- Tanpa Stemming ---")
print("Cleaned:", processed_sample['cleaned'])
print("Filtered Tokens:", processed_sample['filtered_tokens'])

# Dengan stemming
processed_sample_stemmed = nlp.preprocess(sample_text, apply_stemming=True)
print("
--- Dengan Stemming ---")
print("Filtered Tokens (Stemmed):", processed_sample_stemmed['filtered_tokens'])

Sekarang, terapkan preprocessing (dengan stemming) ke seluruh kolom `review`.

In [ ]:
print("Menerapkan preprocessing ke seluruh dataset (ini mungkin memakan waktu)...

df['processed_text'] = df[TEXT_COLUMN].apply(
    lambda x: " ".join(nlp.preprocess(str(x), apply_stemming=True)['filtered_tokens'])
)

print("Preprocessing selesai.")
df[['review', 'processed_text', 'sentiment']].head()

## 5. Ekstraksi Fitur & Pembagian Data

Kita akan membagi data menjadi set training (80%) dan testing (20%), lalu mengubah teks yang sudah diproses menjadi representasi numerik menggunakan **Bag-of-Words (BoW)** dan **TF-IDF**.

In [ ]:
X = df['processed_text']
y = df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Ukuran data training: {len(X_train)}")
print(f"Ukuran data testing: {len(X_test)}")

# Inisialisasi Vectorizers
bow_vectorizer = CountVectorizer()
tfidf_vectorizer = TfidfVectorizer()

# Fit & Transform data training
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform data testing
X_test_bow = bow_vectorizer.transform(X_test)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("
Dimensi matriks BoW (training):", X_train_bow.shape)
print("Dimensi matriks TF-IDF (training):", X_train_tfidf.shape)

## 6. Training & Evaluasi Model

Kita akan melatih tiga model (Naive Bayes, Decision Tree, SVM) pada kedua jenis fitur (BoW dan TF-IDF) dan mengevaluasi performanya.

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(random_state=42)
}

vectorizers = {
    "BoW": (X_train_bow, X_test_bow),
    "TF-IDF": (X_train_tfidf, X_test_tfidf)
}

results = []

for model_name, model in models.items():
    for vec_name, (X_train_vec, X_test_vec) in vectorizers.items():
        print(f"--- Melatih {model_name} dengan {vec_name} ---")
        
        # Training
        model.fit(X_train_vec, y_train)
        
        # Prediksi
        y_pred = model.predict(X_test_vec)
        
        # Evaluasi
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
        
        results.append({
            'Model': model_name,
            'Vectorization': vec_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'y_true': y_test,
            'y_pred': y_pred
        })
        
        print(f"Akurasi: {accuracy:.4f}")

results_df = pd.DataFrame(results)
results_df.sort_values(by='Accuracy', ascending=False, inplace=True)

### Perbandingan Hasil

Mari kita lihat tabel perbandingan performa semua model.

In [ ]:
display(results_df[['Model', 'Vectorization', 'Accuracy', 'Precision', 'Recall', 'F1-Score']])

## 7. Visualisasi Hasil

Visualisasi membantu kita memahami hasil dengan lebih baik. Kita akan membuat:

1.  Grafik batang untuk perbandingan akurasi.
2.  Heatmap Confusion Matrix untuk model terbaik.

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(data=results_df, x='Accuracy', y='Model', hue='Vectorization', orient='h')
plt.title('Perbandingan Akurasi Model')
plt.xlabel('Akurasi')
plt.ylabel('Model')
plt.xlim(0, 1.0)
plt.legend(title='Metode Vektorisasi')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

### Confusion Matrix untuk Model Terbaik

Sekarang kita visualisasikan confusion matrix dari model dengan akurasi tertinggi.

In [ ]:
best_result = results_df.iloc[0]
print(f"Model terbaik: {best_result['Model']} dengan {best_result['Vectorization']}")
print(f"Akurasi: {best_result['Accuracy']:.4f}")

# Laporan Klasifikasi
print("
Classification Report:")
report = classification_report(best_result['y_true'], best_result['y_pred'])
print(report)

# Confusion Matrix
cm = confusion_matrix(best_result['y_true'], best_result['y_pred'])
labels = sorted(y_test.unique())

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix - {best_result["Model"]} ({best_result["Vectorization"]})')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 8. Prediksi pada Teks Baru

Terakhir, kita akan melatih ulang model terbaik menggunakan seluruh data (training + testing) dan menggunakannya untuk memprediksi sentimen dari beberapa review baru.

In [ ]:
# Tentukan model dan vectorizer terbaik
best_model_name = best_result['Model']
best_vec_name = best_result['Vectorization']

final_model = models[best_model_name]
if best_vec_name == 'BoW':
    final_vectorizer = bow_vectorizer
    X_full = final_vectorizer.fit_transform(X) # Fit transform pada seluruh data
else:
    final_vectorizer = tfidf_vectorizer
    X_full = final_vectorizer.fit_transform(X)

# Latih ulang model dengan seluruh data
final_model.fit(X_full, y)

print(f"Model '{best_model_name}' dengan '{best_vec_name}' telah dilatih ulang menggunakan seluruh data.")

# Contoh teks baru
new_reviews = [
    "Aplikasi ini sangat membantu, semuanya jadi lebih mudah!",
    "Sering error dan lambat, tolong diperbaiki.",
    "Biasa saja, tidak ada yang spesial."
]
for review in new_reviews:
    # Preprocess teks baru
    processed_review = " ".join(nlp.preprocess(review, apply_stemming=True)['filtered_tokens'])
    
    # Vectorize teks baru
    vectorized_review = final_vectorizer.transform([processed_review])
    
    # Prediksi
    prediction = final_model.predict(vectorized_review)
    
    print(f"
Review: '{review}'")
    print(f"--> Prediksi Sentimen: {prediction[0]}")

## Kesimpulan

Notebook ini telah menunjukkan alur kerja end-to-end untuk analisis sentimen. Model **{best_result['Model']}** dengan vektorisasi **{best_result['Vectorization']}** memberikan performa terbaik dengan akurasi **{best_result['Accuracy']:.2f}**. Pipeline ini sekarang siap digunakan untuk memprediksi sentimen dari ulasan baru atau diintegrasikan ke dalam aplikasi.